In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../dataset/samples_post_extraction_and_labelled.csv')

In [ ]:
print(df.shape)
cols_to_drop = [col for col in df.columns if col.startswith('diagnostics_')]
cols_to_drop.extend(['sample_id', 'mycn_amplified'])
print(f'Number of cols to drop: {len(cols_to_drop)}')
print(cols_to_drop)

X = df.drop(columns=cols_to_drop)
print(f'Number of features for model: {len(X.columns)}')

y = df['mycn_amplified']

In [ ]:
for col in X.columns:
    print(col, X[col].dtype)

In [ ]:
y.value_counts() # Imbalanced dataset

In [ ]:
# Train validation split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# # Scaling
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_val = scaler.transform(X_val)

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
# # Feature selection
# from sklearn.feature_selection import SelectKBest, f_classif, chi2

# prev_features = len(X_train[0])

# selector = SelectKBest(chi2, k=50)
# X_train = selector.fit_transform(X_train, y_train)
# X_val = selector.transform(X_val)

# post_features = len(X_train[0])
# print(f'{post_features} were selected from {prev_features}')

In [ ]:
# # Dimensionaltiy reduction (TBC)
# from sklearn.decomposition import PCA

# prev_features = len(X_train[0])

# pca = PCA(n_components=20)
# X_train = pca.fit_transform(X_train)
# X_val = pca.transform(X_val)

# post_features = len(X_train[0])
# print(f'{post_features} were selected from {prev_features} for train and {len(X_val[0])} for validation')

In [ ]:
# Resampling
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'X_train changed: {X_train.shape} -> {X_train_res.shape}')
print(f'y_train changed: {y_train.shape} -> {y_train_res.shape}')
print(f'Prev y labels were:\n{y_train.value_counts()}\nNew y labels are:\n{y_train_res.value_counts()}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr = LogisticRegression(max_iter=1000, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(kernel='rbf', random_state=42)
models = {
    'logistic_regression': lr, 
    'random_forest': rf, 
    'support_vector': svm
}

def eval_results(y_actual, predictions): 
    acc = accuracy_score(y_actual, predictions)
    clas_report = classification_report(y_actual, predictions) # Note that F1 score can only be used if true positives = true negatives 
    conf_matrix = confusion_matrix(y_actual, predictions)
    print('Accuracy:', acc)
    print('Classifcation Report:\n', clas_report)
    print('Confusion Matrix:\n', conf_matrix)

for name, model in models.items():
    print(f'\n=== {name} ===')
    model.fit(X_train, y_train)
    model_predictions = model.predict(X_val)
    eval_results(y_val, model_predictions)